# 03 · Station snowpack, snowfall fraction and soil moisture

Station April 1 SWE as a check on the ERA5-Land pattern from notebook 02,
plus two descriptive ERA5-Land columns: the share of winter precipitation that
fell as snow, and summer soil moisture. Every winter is shown, El Niño
winters highlighted (`ENSO_INDEX` from `src/config.py`). Descriptive only: no
tests before the freeze, and no fire data.

Inputs are built by code before this notebook runs:

- `python -m src.stations`: the cleaned station x year table
  (`data/processed/station_swe_apr1.csv`) and the regional index beside
  ERA5-Land (`station_swe_region_year.csv`, also in `results/tables/`).
  NRCS SNOTEL and snow courses (WA, OR, ID, AK), BC and Yukon snow courses,
  CanSWE for Alberta and for BC and Yukon pillows.
- `python -m src.snow_soil`: snowfall fraction (Nov-Mar) and JJA soil
  moisture, 0-28 cm (`snow_soil_region_year.csv`, also in `results/tables/`).

Cleaning rules, thresholds and their measured effects:
`docs/methods_register.md` ("Station snowpack"); reasoning in `DECISIONS.md`
2026-10-05.

In [1]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import pandas as pd

from src.config import (ENSO_INDEX, SNOW_SOIL_TABLE, STATION_INDEX_TABLE, STATION_NORMAL_YEARS,
                        STATION_TABLE)
from src.describe import phase_summary
from src.figures import FIGURES, winter_series, winter_series_columns
from src.stations import agreement, effects_by_group, station_normals

stations = pd.read_csv(STATION_TABLE, parse_dates=["sample_date"])
index = pd.read_csv(STATION_INDEX_TABLE)
snow_soil = pd.read_csv(SNOW_SOIL_TABLE)
normals = station_normals(stations)
NORTH = ["PNW", "BC", "AB", "AK", "YT"]  # regions with stations
print(f"{len(stations):,} station-years, {stations['station_id'].nunique():,} stations; "
      f"{len(normals)} with a {STATION_NORMAL_YEARS[0]}-{STATION_NORMAL_YEARS[1]} normal")

71,759 station-years, 2,180 stations; 935 with a 1991-2020 normal


## Station coverage

Stations per region by source and method (manual snow course or snow
pillow), and how many count toward the index in selected years (a normal
from at least 20 of the 30 years 1991-2020). Alberta has no 2026 value:
CanSWE ends July 2025.

In [2]:
stations.groupby(["region", "source", "method"]).agg(
    stations=("station_id", "nunique"), values=("year", "size"),
    first=("year", "min"), last=("year", "max"))

stations  values  first  last
region source method                               
AB     bc     manual         1      53   1951  2005
       canswe manual       132    5201   1950  2025
              pillow        16     454   1984  2025
AK     canswe manual         4      66   1965  2011
       nrcs   manual       318    8790   1953  2026
              pillow        58    1222   1967  2026
BC     bc     manual       365   12869   1950  2026
       canswe manual        57     205   1950  2025
              pillow       134    2208   1968  2025
       nrcs   manual         1      17   2006  2026
PNW    canswe manual        15     553   1950  2011
       nrcs   manual       800   27998   1950  2026
              pillow       243    9298   1979  2026
YT     canswe manual         4      10   1978  1984
              pillow         8     109   1989  2025
       yukon  manual        82    2706   1965  2026

In [3]:
index[index["region"].isin(NORTH)].pivot(index="year", columns="region", values="n_stations").loc[
    [1951, 1960, 1970, 1980, 1990, 2000, 2010, 2020, 2025, 2026], NORTH].fillna(0).astype(int)

region,PNW,BC,AB,AK,YT
year,,,,,
1951,49,27,4,0,0
1960,99,49,4,7,0
1970,161,118,15,52,2
1980,207,150,80,81,33
1990,365,169,104,125,53
2000,395,204,102,157,56
2010,350,173,109,151,54
2020,309,157,76,111,53
2025,360,138,97,126,52


## Stations against ERA5-Land

Correlation of the yearly series, on % of normal and standardised scales,
over the years the station index exists. Level ratio = median station normal
/ ERA5-Land regional mean, both mm over 1991-2020: stations sit where the
snow is, so a ratio far from 1 is expected and says nothing about the
year-to-year agreement.

In [4]:
agreement(index, normals, stations).round(2)

,region,n_years,first_year,last_year,corr_pct,corr_z,station_normal_median_mm,era5_normal_mm,level_ratio
0,PNW,76,1951,2026,0.94,0.95,361.53,116.71,3.10
1,BC,76,1951,2026,0.79,0.84,451.54,423.66,1.07
2,AB,63,1963,2025,0.84,0.77,75.22,59.74,1.26
3,AK,67,1960,2026,0.78,0.80,152.22,224.73,0.68
4,YT,52,1975,2026,0.84,0.84,150.18,170.06,0.88


In [5]:
winter_series_columns(
    index[index["region"].isin(NORTH)],
    {"stations, median % of normal": "station_swe_pct", "ERA5-Land, % of normal": "era5_swe_pct"},
    "April 1 SWE, % of 1991-2020", ENSO_INDEX,
    FIGURES / f"swe_april1_stations_vs_era5_{ENSO_INDEX.lower()}.png")

WindowsPath('C:/Users/dacek/Documents/Portfolio/hydrowhiplash-elnino26/results/figures/swe_april1_stations_vs_era5_oni.png')

## El Niño differences, side by side

Same years for both sources (those with a station index). In points of
normal and in SDs of each series. No p-values: those come after the
freeze.

In [6]:
common = index[index["region"].isin(NORTH)].dropna(subset=["station_swe_pct"])
cols = ["n_el_nino", "n_strong", "n_neutral", "nino_minus_neutral", "nino_minus_neutral_sd",
        "strong_minus_all_sd", "nino_share_below_median"]
side = pd.concat({name: phase_summary(common, col).set_index("region")[cols]
                  for name, col in [("stations %", "station_swe_pct"), ("ERA5-Land %", "era5_swe_pct")]},
                 axis=1)
strong = {name: common.groupby("region", sort=False).apply(
              lambda d, c=col: d.loc[d["strong"] & (d["phase"] == "el_nino"), c].mean() - d[c].mean())
          for name, col in [("stations %", "station_swe_pct"), ("ERA5-Land %", "era5_swe_pct")]}
for name, s in strong.items():
    side[(name, "strong_minus_all")] = s
side.sort_index(axis=1).round(2).T

region                                 PNW     BC     AB     AK     YT
ERA5-Land % n_el_nino                25.00  25.00  22.00  22.00  18.00
            n_neutral                25.00  25.00  18.00  22.00  16.00
            n_strong                  7.00   7.00   6.00   6.00   5.00
            nino_minus_neutral       -7.98  -7.18 -27.23  -1.07  -4.63
            nino_minus_neutral_sd    -0.17  -0.42  -0.57  -0.07  -0.29
            nino_share_below_median   0.72   0.68   0.73   0.55   0.61
            strong_minus_all        -25.63 -12.96 -32.28  -4.99  -2.06
            strong_minus_all_sd      -0.53  -0.77  -0.68  -0.34  -0.13
stations %  n_el_nino                25.00  25.00  22.00  22.00  18.00
            n_neutral                25.00  25.00  18.00  22.00  16.00
            n_strong                  7.00   7.00   6.00   6.00   5.00
            nino_minus_neutral       -1.33  -5.29 -22.34  -7.52 -12.11
            nino_minus_neutral_sd    -0.03  -0.26  -0.52  -0.32  -0.56
            nino_share_below_median   0.68   0.68   0.68   0.64   0.72
            strong_minus_all         -8.45 -14.28 -38.92  -4.14  -5.53
            strong_minus_all_sd      -0.22  -0.70  -0.91  -0.18  -0.26

## By station elevation

Stations in each region split into thirds by elevation. Strong El Niño
winters against all winters, in points of the median % of normal. Where the
loss is concentrated low, a station network sited high understates the
regional loss that an area-weighted grid mean shows.

In [7]:
enso = index[["year", "phase", "strong"]]
effects_by_group(stations, normals, enso, column="elevation").round(1)

,region,group,elevation_min,elevation_max,stations,nino_minus_neutral,strong_minus_all
0,PNW,1,551.7,1472.2,131,-1.4,-42.3
1,PNW,2,1475.2,1868.4,130,1.0,-4.6
2,PNW,3,1880.6,2801.1,131,-0.3,3.0
3,AK,1,12.2,326.1,53,-1.5,-4.4
4,AK,2,329.2,603.5,53,-8.0,-5.9
5,AK,3,612.6,1286.3,53,-10.4,-1.9
6,AB,1,210.0,700.0,39,-30.2,-58.4
7,AB,2,720.0,1615.0,34,-24.5,-51.0
8,AB,3,1740.0,2380.0,37,-13.0,-12.8
9,BC,1,180.0,1285.0,69,-8.3,-20.9


## Snowfall fraction, every winter

Share of November-March precipitation that fell as snow (ratio of regional
totals). Year Y = November of Y-1 to March of Y.

In [8]:
ss = snow_soil.assign(snowfall_pct=100 * snow_soil["snowfall_frac"])
winter_series(ss.dropna(subset=["snowfall_pct"]), "snowfall_pct", "Snowfall, % of Nov-Mar precip",
              ENSO_INDEX, FIGURES / f"snowfall_fraction_series_{ENSO_INDEX.lower()}.png",
              xlabel="Winter (Nov-Mar); year Y = November Y-1 to March Y")

WindowsPath('C:/Users/dacek/Documents/Portfolio/hydrowhiplash-elnino26/results/figures/snowfall_fraction_series_oni.png')

## Summer soil moisture, every year

June-August mean volumetric soil water, 0-28 cm depth-weighted (m³/m³). Year
Y = summer Y, after winter (Y-1)-Y. 2026 has no complete summer yet.

In [9]:
winter_series(snow_soil.dropna(subset=["soil_moisture"]), "soil_moisture", "Soil water, m³/m³",
              ENSO_INDEX, FIGURES / f"soil_moisture_jja_series_{ENSO_INDEX.lower()}.png",
              xlabel="Summer (Jun-Aug) of year Y, after winter (Y-1)-Y")

WindowsPath('C:/Users/dacek/Documents/Portfolio/hydrowhiplash-elnino26/results/figures/soil_moisture_jja_series_oni.png')

In [10]:
cols = ["n_el_nino", "n_strong", "mean_all", "nino_minus_neutral", "nino_minus_neutral_pct",
        "nino_minus_neutral_sd", "strong_minus_all_pct", "strong_minus_all_sd"]
pd.concat({v: phase_summary(snow_soil.dropna(subset=[v]), v).set_index("region")[cols]
           for v in ["snowfall_frac", "soil_moisture"]}, axis=1).round(3).T

region                                   PNW      BC      AB      AK      YT  \
snowfall_frac n_el_nino               25.000  25.000  25.000  25.000  25.000   
              n_strong                 7.000   7.000   7.000   7.000   7.000   
              mean_all                 0.458   0.782   0.920   0.832   0.991   
              nino_minus_neutral      -0.016  -0.018  -0.009  -0.001  -0.000   
              nino_minus_neutral_pct  -3.588  -2.251  -0.945  -0.091  -0.050   
              nino_minus_neutral_sd   -0.240  -0.410  -0.328  -0.017  -0.094   
              strong_minus_all_pct   -12.489  -3.682  -0.203  -1.811  -0.017   
              strong_minus_all_sd     -0.837  -0.671  -0.071  -0.339  -0.032   
soil_moisture n_el_nino               25.000  25.000  25.000  25.000  25.000   
              n_strong                 7.000   7.000   7.000   7.000   7.000   
              mean_all                 0.213   0.326   0.350   0.318   0.257   
              nino_minus_neutral       0.000   0.001   0.001   0.002  -0.000   
              nino_minus_neutral_pct   0.229   0.360   0.246   0.486  -0.174   
              nino_minus_neutral_sd    0.033   0.068   0.048   0.111  -0.034   
              strong_minus_all_pct     0.075  -3.368  -0.186   2.927  -0.801   
              strong_minus_all_sd      0.011  -0.640  -0.036   0.666  -0.156   

region                                NorCal   SoCal  
snowfall_frac n_el_nino               25.000  25.000  
              n_strong                 7.000   7.000  
              mean_all                 0.309   0.063  
              nino_minus_neutral       0.006  -0.006  
              nino_minus_neutral_pct   2.073  -9.784  
              nino_minus_neutral_sd    0.103  -0.162  
              strong_minus_all_pct   -10.387 -45.322  
              strong_minus_all_sd     -0.518  -0.752  
soil_moisture n_el_nino               25.000  25.000  
              n_strong                 7.000   7.000  
              mean_all                 0.209   0.122  
              nino_minus_neutral       0.005   0.006  
              nino_minus_neutral_pct   2.358   4.787  
              nino_minus_neutral_sd    0.458   0.628  
              strong_minus_all_pct     3.248   6.664  
              strong_minus_all_sd      0.631   0.875